# Computer Vision Cart Checkout

Upload a composed photo of multiple grocery items across multiple classes; the system **detects** each item, **recognizes** what it is, **matches** it to a catalog, and **totals** the basket from a price table.

This notebook implements the full five-step pipeline on the
[GroceryStoreDataset](https://github.com/marcusklasson/GroceryStoreDataset) (81 fine classes, 43 coarse
categories, 3 types — **no bounding boxes provided**, so we manufacture them).

**How the steps connect — each loads the previous checkpoint from Google Drive and saves its own, so you can stop and resume at any step:**

| Step | Does | Key idea |
|------|------|----------|
| 1 | Labels table + DINOv2 catalog gallery + price table | Freeze DINOv2; one clean iconic shot per class anchors the catalog |
| 2 | Auto-box source images, compose scenes, fine-tune YOLO11 | Detector finds *where* only — trained as a **single `product` class** |
| 3 | Train recognizer on frozen features + diagnose | Benchmark DINOv2 vs YOLO features, keep the winner; name weak classes |
| 4 | Augment weak classes, re-tune detector | Any generative aug that gives a **real** validation lift |
| 5 | Gradio self-checkout demo | Detect → recognize → catalog-match → total |

**Runtime:** Google Colab with a **GPU** runtime (Runtime → Change runtime type → T4 GPU). Steps 1–4 train models; Step 5 only loads checkpoints.

> **One honest caveat, up front:** every bounding box here is *manufactured* by Step 2's auto-boxer, so mAP measures agreement-with-the-autoboxer, not ground truth. Step 2 includes an optional hand-labeled gold set so you have at least one trustworthy number.

---
# Step 1 — Catalog & labels foundation

**Goal:** turn the pile of product photos into two reusable assets — a clean **labels table** and a searchable **catalog gallery** on frozen DINOv2 embeddings — plus a **price table**. Everything is saved to a `CvCart` folder in Google Drive as the first checkpoint.

Why frozen DINOv2: its features are strong off-the-shelf, and freezing keeps this step fast, deterministic, and reusable both as the catalog matcher (Step 5) and as a candidate recognizer backbone (Step 3).

### 1.0 — Runtime setup and the checkpoint bundle

We mount Google Drive (stores the small cross-part bundle, not the image dataset) and define a tiny `Bundle`
helper that reads/writes tables and arrays and keeps a `manifest.json` honest, so every later part can assert
its inputs exist before running.

In [8]:
# Install only what this notebook needs. (Ultralytics pulls in torch/torchvision on Colab.)
%pip install -q ultralytics gradio qrcode

import os, json, pathlib, subprocess, random
import numpy as np
import pandas as pd

# ---- Google Drive: holds the cross-part checkpoint bundle ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BUNDLE_DIR = '/content/drive/MyDrive/CvCart'
except Exception:
    # Local fallback so the notebook still runs outside Colab.
    BUNDLE_DIR = os.path.abspath('./CvCart')
    print('Not in Colab — using local bundle dir:', BUNDLE_DIR)

SEED = 0
random.seed(SEED); np.random.seed(SEED)


class Bundle:
    '''Small Google Drive folder that carries artifacts from one part to the next.'''
    def __init__(self, root):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.manifest = {'version': 1, 'class_list': [], 'artifacts': {}}
        self.load()

    def put_table(self, name, df):
        df.to_csv(self.root / name, index=False); self._note(name); return self

    def get_table(self, name):
        return pd.read_csv(self.root / name)

    def put_array(self, name, arr):
        np.save(self.root / name, arr); self._note(name if name.endswith('.npy') else name + '.npy'); return self

    def get_array(self, name):
        p = self.root / name
        return np.load(p if p.suffix == '.npy' else p.with_suffix('.npy'))

    def has(self, name):
        return (self.root / name).exists() or (self.root / (name + '.npy')).exists()

    def _note(self, name):
        self.manifest['artifacts'][name] = True

    def save(self):
        (self.root / 'manifest.json').write_text(json.dumps(self.manifest, indent=2))
        print('[bundle] saved ->', self.root); return self

    def load(self):
        p = self.root / 'manifest.json'
        if p.exists():
            self.manifest = json.loads(p.read_text())
        return self


b = Bundle(BUNDLE_DIR)
print('bundle ready at', b.root, '| existing artifacts:', list(b.manifest['artifacts']))

### 1.1 — Clone the dataset and build the labels table

We clone GroceryStoreDataset and parse `classes.csv` together with the three split files into one **labels
table**. Two dataset-specific gotchas, both handled below:

- The split files (`train.txt` / `val.txt` / `test.txt`) are `path, fine_id, coarse_id` — **fine ID first**,
  which is the opposite of the intuitive order and a classic source of silent label bugs.
- Class IDs in `classes.csv` are the source of truth for names; we join on them rather than trusting folder
  names alone.

In [ ]:
GROCERY_URL = 'https://github.com/marcusklasson/GroceryStoreDataset'
DATA_ROOT = pathlib.Path('/content/GroceryStoreDataset')

if (DATA_ROOT / 'dataset' / 'classes.csv').exists():
    print('using cached dataset:', DATA_ROOT)
else:
    print('cloning', GROCERY_URL, '...')
    subprocess.run(['git', 'clone', '--depth', '1', GROCERY_URL, str(DATA_ROOT)], check=True)
DATASET = DATA_ROOT / 'dataset'
assert (DATASET / 'classes.csv').exists(), 'clone incomplete — check Colab network and rerun'

# --- classes.csv: normalize the verbose headers to short names ---
classes = pd.read_csv(DATASET / 'classes.csv')
classes.columns = [c.split(' (')[0].strip().lower().replace(' ', '_') for c in classes.columns]
# columns: class_name, class_id, coarse_class_name, coarse_class_id, iconic_image_path, product_description_path
FINE_NAMES = classes.sort_values('class_id')['class_name'].tolist()
N_FINE = len(FINE_NAMES)
id2fine = dict(zip(classes.class_id, classes.class_name))
id2coarse = dict(zip(classes.class_id, classes.coarse_class_name))
print(f'{N_FINE} fine classes, {classes.coarse_class_name.nunique()} coarse categories')

def load_split(split):
    df = pd.read_csv(DATASET / f'{split}.txt', header=None,
                     names=['rel_path', 'fine_id', 'coarse_id'], skipinitialspace=True)
    df['rel_path'] = df['rel_path'].str.strip()
    df['split'] = split
    df['abs_path'] = df['rel_path'].apply(lambda p: str(DATASET / p))
    df['type'] = df['rel_path'].str.split('/').str[1]          # Fruit / Vegetables / Packages
    df['fine'] = df['fine_id'].map(id2fine)
    df['coarse'] = df['fine_id'].map(id2coarse)                # derive coarse from fine id (authoritative)
    return df

labels_table = pd.concat([load_split(s) for s in ('train', 'val', 'test')], ignore_index=True)
print('labels table:', labels_table.split.value_counts().to_dict())

# Class imbalance is the reason Step 4 exists — surface it now.
per_class = labels_table[labels_table.split == 'train'].groupby('fine').size()
print(f'train images per fine class -> min {per_class.min()}, median {int(per_class.median())}, max {per_class.max()}')

b.manifest['class_list'] = FINE_NAMES
b.put_table('labels_table.csv', labels_table)
print('saved labels_table.csv')

### 1.2 — Frozen DINOv2 catalog gallery

We load a **frozen** DINOv2 backbone and embed the **81 iconic images** (one clean studio shot per fine
class) into a matrix of L2-normalized vectors. This gallery is the "what does each product look like"
reference used for catalog matching in Step 5, and DINOv2 is also a candidate recognizer backbone in Step 3.

Keep this preprocessing identical wherever DINOv2 is used later, or catalog matches will silently drift.

In [ ]:
import torch
from PIL import Image
import torchvision.transforms as T

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

# Frozen DINOv2 (small ViT-S/14). torch.hub caches after first download.
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(DEVICE).eval()
for p in dino.parameters():
    p.requires_grad_(False)

# DINOv2 expects 14-multiple sizes; 224 is standard. Same transform used in Steps 3 & 5.
dino_tf = T.Compose([
    T.Resize(256), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def dino_embed(pil_img):
    x = dino_tf(pil_img.convert('RGB')).unsqueeze(0).to(DEVICE)
    feat = dino(x)                        # (1, 384) CLS embedding for ViT-S/14
    feat = torch.nn.functional.normalize(feat, dim=1)
    return feat.squeeze(0).cpu().numpy()

# Embed one iconic image per fine class -> the catalog gallery.
catalog_rows, catalog_vecs = [], []
for _, r in classes.sort_values('class_id').iterrows():
    icon = DATASET / r['iconic_image_path'].lstrip('/')
    if not icon.exists():
        print('missing iconic for', r['class_name'], '- skipping'); continue
    catalog_vecs.append(dino_embed(Image.open(icon)))
    catalog_rows.append({'row': len(catalog_rows), 'fine_id': r['class_id'], 'fine': r['class_name'],
                         'coarse': r['coarse_class_name'], 'iconic_path': str(icon)})

catalog_embeddings = np.stack(catalog_vecs).astype('float32')
catalog_index = pd.DataFrame(catalog_rows)
print('catalog gallery:', catalog_embeddings.shape, '- one vector per class')

b.put_array('catalog_embeddings.npy', catalog_embeddings)
b.put_table('catalog_index.csv', catalog_index)
print('saved catalog embeddings + index')

### 1.3 — Price table and save the Step 1 checkpoint

A random-but-reproducible USD price per fine class (fixed seed). Then we write the manifest — Step 1 done.

In [ ]:
rng = np.random.default_rng(SEED)
price_table = pd.DataFrame({
    'fine_id': classes.class_id,
    'fine': classes.class_name,
    'price_usd': np.round(rng.uniform(0.50, 9.99, size=len(classes)), 2),
})
b.put_table('price_table.csv', price_table)

b.save()
print('Step 1 complete. Bundle artifacts:', list(b.manifest['artifacts']))
price_table.head()

---
# Step 2 — Auto-boxing, scene composition, YOLO11 detector

**Goal:** manufacture trustworthy boxes, compose realistic multi-item scenes, fine-tune YOLO11 to find
products (not classify them), and crop detections back out for Step 3.

The dataset has **no boxes**, and its photos come in three kinds that break naive boxing:
1. **Held / single-subject** — one clear foreground item (often a hand + busy background).
2. **Cluttered piles** — dozens of the *same* item filling the frame; sometimes a distractor object.
3. **Shelf / context** — the target product surrounded by competitors.

Our approach: isolate **one** reliable box per image behind an explicit **accept/reject gate**, **discard**
anything that can't be boxed cleanly, then **compose multi-item scenes** so the detector trains on the
cluttered-basket distribution it will actually face — as a **single `product` class**.

### 2.0 — Load the Step 1 checkpoint

Resume-safe: we reopen the bundle and reload the labels table, so Step 2 can run in a fresh runtime.

In [ ]:
# (Re)load bundle + Step 1 outputs so this step runs standalone after a runtime reset.
import os, pathlib, random
import numpy as np, pandas as pd
from PIL import Image, ImageDraw

assert 'b' in dir(), 'Run Step 1 setup cell first (defines the Bundle `b`).'
labels_table = b.get_table('labels_table.csv')
FINE_NAMES = b.manifest['class_list']
print('loaded labels table:', len(labels_table), 'images |', len(FINE_NAMES), 'classes')

# Speed knob: cap images considered per class during boxing (0 = use all). Raise for a fuller run.
PER_CLASS_CAP = int(os.environ.get('CVCART_PER_CLASS', '50'))
work = (labels_table.groupby(['split', 'fine'], group_keys=False).head(PER_CLASS_CAP)
        if PER_CLASS_CAP > 0 else labels_table).reset_index(drop=True)
print('working set for boxing:', len(work), f'(cap {PER_CLASS_CAP}/class/split)')

### 2.1 — Auto-boxing with an explicit accept/reject gate

For each source image we generate one candidate box (class-agnostic — a pretrained YOLO11 tells us *where*,
its class guess is discarded; GrabCut is the fallback when YOLO finds nothing), then **keep it only if it
passes all gates**:

- **Area sanity** — not a sliver, not near-full-frame (handles held/shelf shots; the `<~40%` upper bound).
- **Focus** — crop sharpness (Laplacian variance) above a floor, so we don't box a blurry background object.
- **Centrality** — the candidate is meaningfully central, where the labelled product usually sits.

For **cluttered piles** this naturally yields *one representative in-focus instance* rather than the whole
heap. If nothing clears the gate, the image is **discarded** and logged with a reason — dropping bad samples
is intentional, because poor boxes hurt both training and evaluation.

In [ ]:
import cv2
from ultralytics import YOLO
from tqdm.auto import tqdm

autoboxer = YOLO('yolo11s.pt')   # localizer only; its 80 COCO classes are irrelevant, we keep geometry

# --- Gate thresholds ---
MIN_AREA, MAX_AREA = 0.05, 0.60   # slivers / near-full-frame reject; per-type caps below refine this
TYPE_MAX_AREA = {'Fruit': 0.40, 'Vegetables': 0.45, 'Packages': 0.60}  # fruit subjects are smaller in frame
FOCUS_FLOOR = 40.0                # Laplacian variance floor; blurry crops rejected
CENTER_CROP_FRAC = 0.6            # size of the GrabCut/last-resort central region

def _sharpness(gray):
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def grabcut_box(img, max_area=MAX_AREA):
    '''Class-agnostic fallback: central-rect GrabCut, pick the best-sized central foreground blob.'''
    H0, W0 = img.shape[:2]
    if max(H0, W0) > 320:                      # downscale first -> GrabCut is far faster; coords stay normalized
        s_ = 320.0 / max(H0, W0); img = cv2.resize(img, (int(W0 * s_), int(H0 * s_)))
    H, W = img.shape[:2]; m = 0.08
    rect = (int(W*m), int(H*m), int(W*(1-2*m)), int(H*(1-2*m)))
    mask = np.zeros((H, W), np.uint8)
    bgd, fgd = np.zeros((1,65),np.float64), np.zeros((1,65),np.float64)
    try:
        cv2.grabCut(img, mask, rect, bgd, fgd, 4, cv2.GC_INIT_WITH_RECT)
    except Exception:
        return None
    fg = np.where((mask==cv2.GC_FGD)|(mask==cv2.GC_PR_FGD),255,0).astype(np.uint8)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((5,5),np.uint8))
    cnts,_ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cx, cy, maxd = W/2, H/2, ((W/2)**2+(H/2)**2)**0.5
    best, best_s = None, -1
    for c in cnts:
        x,y,w,h = cv2.boundingRect(c); frac=(w*h)/(W*H)
        if frac < MIN_AREA or frac > max_area: continue
        dist = (((x+w/2)-cx)**2 + ((y+h/2)-cy)**2)**0.5
        s = frac*(0.4+0.6*(1-dist/maxd))
        if s > best_s: best, best_s = (x,y,w,h), s
    if best is None: return None
    x,y,w,h = best
    return (x/W, y/H, w/W, h/H)   # xywh (top-left) normalized

def auto_box(path, max_area=MAX_AREA):
    '''Return (x,y,w,h, source, focus) normalized top-left box, or None if it fails the gate.'''
    img = cv2.imread(path)
    if img is None: return None
    H, W = img.shape[:2]; cx, cy = W/2, H/2; maxd = (cx**2+cy**2)**0.5
    res = autoboxer.predict(path, conf=0.10, agnostic_nms=True, verbose=False)[0]
    cands = []
    for bx in res.boxes:
        x1,y1,x2,y2 = bx.xyxy[0].tolist(); bw,bh = x2-x1, y2-y1
        area = (bw*bh)/(W*H)
        if bw<4 or bh<4 or area<MIN_AREA or area>max_area: continue
        gray = cv2.cvtColor(img[int(y1):int(y2), int(x1):int(x2)], cv2.COLOR_BGR2GRAY)
        focus = _sharpness(gray)
        if focus < FOCUS_FLOOR: continue                       # focus gate
        dist = (((x1+x2)/2-cx)**2 + ((y1+y2)/2-cy)**2)**0.5
        central = 1 - dist/maxd
        score = (0.3+0.7*float(bx.conf[0])) * (0.3+0.7*central)
        cands.append((score, (x1/W, y1/H, bw/W, bh/H), focus))
    if cands:
        _, box, focus = max(cands, key=lambda t: t[0])
        return (*box, 'yolo', focus)
    fb = grabcut_box(img, max_area=max_area)                   # YOLO gave nothing -> GrabCut
    if fb is not None:
        x,y,w,h = fb
        gray = cv2.cvtColor(img[int(y*H):int((y+h)*H), int(x*W):int((x+w)*W)], cv2.COLOR_BGR2GRAY)
        f = _sharpness(gray)
        if f >= FOCUS_FLOOR:
            return (*fb, 'grabcut', f)
    return None                                                # discard

records, kept, dropped = [], 0, 0
for r in tqdm(list(work.itertuples(index=False)), total=len(work), desc='auto-boxing'):
    res = auto_box(r.abs_path, max_area=TYPE_MAX_AREA.get(getattr(r, 'type', ''), MAX_AREA))
    if res is None:
        dropped += 1
        records.append({'abs_path': r.abs_path, 'fine': r.fine, 'split': r.split,
                        'kept': False, 'reason': 'no_box_passed_gate'})
        continue
    x,y,w,h,src,focus = res
    kept += 1
    records.append({'abs_path': r.abs_path, 'fine': r.fine, 'coarse': r.coarse, 'split': r.split,
                    'type': getattr(r, 'type', None), 'kept': True, 'reason': src,
                    'x': x, 'y': y, 'w': w, 'h': h, 'focus': round(focus,1)})

boxes_manifest = pd.DataFrame(records)
b.put_table('boxes_manifest.csv', boxes_manifest); b.save()
print(f'auto-boxing: kept {kept}, discarded {dropped} ({100*dropped/max(1,kept+dropped):.0f}% dropped)')
print('kept box sources:', boxes_manifest[boxes_manifest.kept].reason.value_counts().to_dict())
# Flag classes that lost too many samples — Step 4 should prioritize these.
kept_per_class = boxes_manifest[boxes_manifest.kept].groupby('fine').size()
thin = kept_per_class[kept_per_class < 5]
if len(thin): print('WARN thin classes after boxing (<5 kept):', list(thin.index))

### 2.2 — Sanity-check the boxes

One accepted box per source kind, so you can *see* the gate working before trusting thousands of labels.

In [14]:
import matplotlib.pyplot as plt
show = boxes_manifest[boxes_manifest.kept].groupby('reason').head(2)
n = len(show)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
for ax, r in zip(np.atleast_1d(axes), show.itertuples(index=False)):
    im = Image.open(r.abs_path).convert('RGB'); W,H = im.size
    d = ImageDraw.Draw(im)
    d.rectangle([r.x*W, r.y*H, (r.x+r.w)*W, (r.y+r.h)*H], outline='lime', width=4)
    ax.imshow(im); ax.set_title(f'[{r.reason}] {r.fine}\nfocus={r.focus}'); ax.axis('off')
plt.tight_layout(); plt.show()

### 2.3 — Compose multi-item scenes with proportional boxes

The real task is a cluttered basket, so the detector must train on cluttered scenes — not single items (the
key flaw in earlier attempts). We paste accepted crops onto blank canvases: mostly multi-item (2–6 products,
mixed classes), some single-item. Each pasted crop's ground-truth box is its **subject box scaled
proportionally into the pasted region**, so boxes hug the product, not the paste rectangle. Scenes are split
train/val so the detector validates on the same distribution it trains on.

In [15]:
SCENES_DIR = pathlib.Path('/content/scenes')
for s in ('train', 'val'):
    (SCENES_DIR/'images'/s).mkdir(parents=True, exist_ok=True)
    (SCENES_DIR/'labels'/s).mkdir(parents=True, exist_ok=True)

from PIL import ImageFilter, ImageEnhance
kept_df = boxes_manifest[boxes_manifest.kept].reset_index(drop=True)
by_class = {f: g for f, g in kept_df.groupby('fine')}
bg_pool = kept_df.abs_path.tolist()          # real photos reused (blurred) as cluttered backgrounds
rng = np.random.default_rng(SEED)
CANVAS = 640
N_SCENES = int(os.environ.get('CVCART_SCENES', '160'))

def subject_crop(row):
    """Crop the source image to its auto-boxer subject box -> a tight product, minimal source background."""
    im = Image.open(row.abs_path).convert('RGB'); W, H = im.size
    x1, y1 = int(row.x * W), int(row.y * H)
    x2, y2 = int((row.x + row.w) * W), int((row.y + row.h) * H)
    return im.crop((max(0, x1), max(0, y1), min(W, x2), min(H, y2)))

def random_background():
    """A cluttered, blurred real-photo background (so the detector stops keying on flat grey)."""
    bg = Image.open(bg_pool[int(rng.integers(0, len(bg_pool)))]).convert('RGB')
    bg = bg.resize((CANVAS, CANVAS)).filter(ImageFilter.GaussianBlur(8))
    return ImageEnhance.Brightness(bg).enhance(float(rng.uniform(0.7, 1.1)))

def _iou(a, b):
    ax, ay, aw, ah = a; bx, by, bw, bh = b
    ix1, iy1 = max(ax, bx), max(ay, by); ix2, iy2 = min(ax+aw, bx+bw), min(ay+ah, by+bh)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1); u = aw*ah + bw*bh - inter
    return inter/u if u else 0.0

scene_map = {}
for i in range(N_SCENES):
    split = 'val' if i % 5 == 0 else 'train'
    canvas = np.array(random_background())
    n_items = 1 if i % 6 == 0 else int(rng.integers(3, 9))          # denser than before
    placed, gt = [], []
    for _ in range(n_items):
        fine = rng.choice(list(by_class))
        row = by_class[fine].iloc[int(rng.integers(0, len(by_class[fine])))]
        crop = subject_crop(row).resize((int(rng.integers(60, 200)),) * 2)   # smaller + wider scale range
        crop = ImageEnhance.Brightness(crop).enhance(float(rng.uniform(0.75, 1.25)))
        if rng.random() < 0.5:
            crop = crop.rotate(float(rng.uniform(-20, 20)), expand=True, fillcolor=(245, 245, 245))
        cw, ch = crop.size
        if cw >= CANVAS or ch >= CANVAS:
            continue
        px = py = 0
        for _try in range(30):                                       # allow some occlusion (IoU < 0.35)
            px = int(rng.integers(0, CANVAS - cw)); py = int(rng.integers(0, CANVAS - ch))
            if all(_iou((px, py, cw, ch), b) < 0.35 for b in placed):
                break
        canvas[py:py+ch, px:px+cw] = np.array(crop)                  # paste the TIGHT crop
        placed.append((px, py, cw, ch))
        gt.append(((px+cw/2)/CANVAS, (py+ch/2)/CANVAS, cw/CANVAS, ch/CANVAS, fine))  # EXACT box = paste rect
    Image.fromarray(canvas).save(SCENES_DIR/'images'/split/f'scene_{i:03d}.jpg')
    lines = [f'0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}' for (xc, yc, w, h, _) in gt]   # single 'product' class
    (SCENES_DIR/'labels'/split/f'scene_{i:03d}.txt').write_text('\n'.join(lines) + '\n')
    scene_map[i] = (split, gt)

(SCENES_DIR/'data.yaml').write_text(
    f'path: {SCENES_DIR}\ntrain: images/train\nval: images/val\nnames:\n  0: product\n')
print('composed', len(scene_map), 'REALISTIC scenes (tight crops on real backgrounds, occlusion, EXACT boxes)')

# Preview 3 scenes with their GT boxes.
ids = list(scene_map)[:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, i in zip(np.atleast_1d(axes), ids):
    split, gt = scene_map[i]
    im = Image.open(SCENES_DIR/'images'/split/f'scene_{i:03d}.jpg').convert('RGB'); d = ImageDraw.Draw(im)
    for xc, yc, w, h, _ in gt:
        d.rectangle([(xc-w/2)*CANVAS, (yc-h/2)*CANVAS, (xc+w/2)*CANVAS, (yc+h/2)*CANVAS], outline='lime', width=3)
    ax.imshow(im); ax.set_title(f'scene_{i:03d} ({split}) - {len(gt)} items'); ax.axis('off')
plt.tight_layout(); plt.show()

### 2.4 — Fine-tune YOLO11 as a single-class detector

The detector's only job is **localization**; classification is Step 3. Training at **one class** removes the
fine-grained burden that pins mAP in the low 0.2s when you (wrongly) train 81 classes on ~30 images each. We
train from `yolo11s.pt`, validate on held-out scenes, and report mAP honestly.

In [16]:
import torch
EPOCHS = int(os.environ.get('CVCART_EPOCHS', '40'))
IMGSZ  = int(os.environ.get('CVCART_IMGSZ', '768'))
det = YOLO('yolo11s.pt')
res = det.train(data=str(SCENES_DIR/'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ,
                patience=15, seed=SEED, device=0 if torch.cuda.is_available() else 'cpu',
                mosaic=1.0, mixup=0.15, degrees=10, scale=0.5, perspective=0.0005, flipud=0.5, verbose=False)
try:
    print(f'VAL mAP50: {float(res.box.map50):.3f} | mAP50-95: {float(res.box.map):.3f}')
except Exception:
    print('training done; see runs/ for metrics')

import shutil
detector_path = os.path.join(getattr(res, 'save_dir', 'runs/detect/train'), 'weights', 'best.pt')
shutil.copy(detector_path, b.root/'detector.pt'); b._note('detector.pt'); b.save()
detector = YOLO(str(b.root/'detector.pt'))
print('saved detector.pt ->', b.root)

### 2.5 — Crop detections back out for Step 3

We run the fine-tuned detector over every scene, crop each detected box, and **recover its fine label** by
IoU-matching the detection against the composed ground-truth boxes (which remember their class). These
real-shaped detector-output crops — not clean catalog photos — are what Step 3's recognizer learns from.

In [ ]:
from tqdm.auto import tqdm
crops_dir = b.root/'crops'; crops_dir.mkdir(exist_ok=True)

def iou_xywh(a, b_):
    ax,ay,aw,ah=a; bx,by,bw,bh=b_
    ix1,iy1=max(ax,bx),max(ay,by); ix2,iy2=min(ax+aw,bx+bw),min(ay+ah,by+bh)
    iw,ih=max(0,ix2-ix1),max(0,iy2-iy1); inter=iw*ih; union=aw*ah+bw*bh-inter
    return inter/union if union else 0.0

rows = []
for i,(split, gt) in tqdm(scene_map.items(), total=len(scene_map), desc='cropping detections'):
    p = SCENES_DIR/'images'/split/f'scene_{i:03d}.jpg'
    arr = np.array(Image.open(p).convert('RGB'))
    det_res = detector(str(p), verbose=False)[0]
    # GT in pixel xywh (top-left) for IoU matching
    gt_px = [((xc-w/2)*CANVAS,(yc-h/2)*CANVAS,w*CANVAS,h*CANVAS, fine) for (xc,yc,w,h,fine) in gt]
    for j, box in enumerate(det_res.boxes.xyxy.cpu().numpy()):
        x1,y1,x2,y2 = [int(v) for v in box]; dw,dh = x2-x1, y2-y1
        if dw<6 or dh<6: continue
        best = max(gt_px, key=lambda g: iou_xywh((x1,y1,dw,dh), g[:4]), default=None)
        if best is None or iou_xywh((x1,y1,dw,dh), best[:4]) < 0.2: continue   # unmatched -> skip
        fine = best[4]
        cp = crops_dir/f'scene{i:03d}_{j}.jpg'
        Image.fromarray(arr[y1:y2, x1:x2]).save(cp)
        rows.append({'crop_path': str(cp), 'fine': fine, 'split': split})

crops_manifest = pd.DataFrame(rows)
b.put_table('crops_manifest.csv', crops_manifest); b.save()
print('crops for Step 3:', len(crops_manifest), '| per split:', crops_manifest.split.value_counts().to_dict())
print('Step 2 complete.')

### 2.6 — (Optional) hand-labeled gold set

Because every box above is model-generated, mAP measures *agreement with the autoboxer*, not truth. For one
trustworthy number, hand-label ~30–50 real scenes (e.g. with Label Studio, single `product` class) and
evaluate `detector.pt` against them. This cell is a stub showing where that evaluation slots in — point it at
a folder of gold images + YOLO-format labels when you have them.

In [ ]:
GOLD_YAML = os.environ.get('CVCART_GOLD_YAML', '')   # e.g. '/content/gold/data.yaml'
if GOLD_YAML and os.path.exists(GOLD_YAML):
    gm = detector.val(data=GOLD_YAML, verbose=False)
    print(f'GOLD mAP50: {float(gm.box.map50):.3f} | mAP50-95: {float(gm.box.map):.3f}')
else:
    print('No gold set provided (set CVCART_GOLD_YAML). Skipping — headline mAP is autoboxer-agreement only.')

---
# Step 3 — Recognizer + diagnosis

**Goal:** learn *what* each detected crop is with a tiny linear head on **frozen** features, then **diagnose**
it — because a checkout that silently misreads items is worse than useless.

Per the plan, we **benchmark two backbones** (DINOv2 from Step 1 vs YOLO11's backbone) and keep whichever
scores higher on held-out crops. Then we produce per-class accuracy, a confusion matrix, and explanation
heatmaps, and emit a **weak-class report** that Step 4 consumes.

### 3.0 — Load the Step 2 checkpoint

Reload the crops manifest and the detector (its backbone is one of the two feature sources we benchmark).

In [ ]:
import os, pathlib
import numpy as np, pandas as pd, torch
from PIL import Image

assert 'b' in dir(), 'Run Step 1 setup cell first.'
crops_manifest = b.get_table('crops_manifest.csv')
FINE_NAMES = b.manifest['class_list']
fine2id = {n: i for i, n in enumerate(FINE_NAMES)}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('crops:', len(crops_manifest), '| classes present:', crops_manifest.fine.nunique())

# Ensure DINOv2 is available even in a fresh runtime (Step 1 may not have run this session).
if 'dino' not in dir():
    import torchvision.transforms as T
    dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(DEVICE).eval()
    for p in dino.parameters(): p.requires_grad_(False)
    dino_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                         T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    @torch.no_grad()
    def dino_embed(pil):
        x = dino_tf(pil.convert('RGB')).unsqueeze(0).to(DEVICE)
        return torch.nn.functional.normalize(dino(x), dim=1).squeeze(0).cpu().numpy()
from ultralytics import YOLO
if 'detector' not in dir():
    detector = YOLO(str(b.root/'detector.pt'))

### 3.1 — Extract frozen features from both backbones

For every crop we compute (a) a DINOv2 CLS embedding and (b) a pooled YOLO11 backbone feature. Both
extractors are frozen — we only ever train a small linear head on top, which is fast and hard to overfit on
limited data.

In [ ]:
import torch.nn.functional as F

# --- YOLO backbone feature extractor: hook the pooled features before the detect head ---
yolo_model = detector.model.to(DEVICE).eval()
@torch.no_grad()
def yolo_embed(pil):
    import torchvision.transforms as T
    tf = T.Compose([T.Resize((224,224)), T.ToTensor()])
    x = tf(pil.convert('RGB')).unsqueeze(0).to(DEVICE)
    feats = []
    h = yolo_model.model[-2].register_forward_hook(lambda m,i,o: feats.append(o))
    try:
        yolo_model(x)
    finally:
        h.remove()
    f = feats[-1]
    if isinstance(f, (list, tuple)): f = f[-1]
    f = F.adaptive_avg_pool2d(f, 1).flatten(1)           # global-average-pool to a vector
    return F.normalize(f, dim=1).squeeze(0).cpu().numpy()

def build_matrix(embed_fn):
    X, y, sp = [], [], []
    for r in crops_manifest.itertuples(index=False):
        try:
            X.append(embed_fn(Image.open(r.crop_path)))
            y.append(fine2id[r.fine]); sp.append(r.split)
        except Exception:
            continue
    return np.stack(X).astype('float32'), np.array(y), np.array(sp)

print('extracting DINOv2 features...'); Xd, yd, spd = build_matrix(dino_embed)
print('extracting YOLO features...');   Xy, yy, spy = build_matrix(yolo_embed)
print('DINOv2:', Xd.shape, '| YOLO:', Xy.shape)

### 3.2 — Train + benchmark a linear head on each backbone

A single linear layer (softmax over 81 classes) per backbone, trained on the train-split crops and scored on
val-split crops. We keep the **winning backbone** and record it in the checkpoint so Step 5 uses the matching
extractor. (Crops from Step 2 were split train/val by their source scene, so this is an honest comparison.)

In [ ]:
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import label_binarize

def train_linear(X, y, sp, epochs=200, lr=1e-2):
    tr, va = sp == 'train', sp == 'val'
    if va.sum() == 0:                                   # fall back to a random split if val is empty
        idx = np.arange(len(y)); np.random.default_rng(0).shuffle(idx)
        cut = int(0.8 * len(idx)); tr = np.zeros(len(y), bool); va = np.zeros(len(y), bool)
        tr[idx[:cut]] = True; va[idx[cut:]] = True
    Xt = torch.tensor(X[tr]).to(DEVICE); yt = torch.tensor(y[tr]).long().to(DEVICE)
    Xv = torch.tensor(X[va]).to(DEVICE); yv = torch.tensor(y[va]).long().to(DEVICE)
    head = torch.nn.Linear(X.shape[1], len(FINE_NAMES)).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-4)
    lossf = torch.nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad(); loss = lossf(head(Xt), yt); loss.backward(); opt.step()
    with torch.no_grad():
        logits = head(Xv); prob = torch.softmax(logits, 1).cpu().numpy()
        pred = logits.argmax(1); acc = (pred == yv).float().mean().item()
    # classification mean Average Precision (macro, one-vs-rest) over classes present in val
    yv_np = yv.cpu().numpy()
    Yb = label_binarize(yv_np, classes=np.arange(len(FINE_NAMES)))
    seen = Yb.sum(0) > 0
    try:
        mAP = float(average_precision_score(Yb[:, seen], prob[:, seen], average='macro'))
    except Exception:
        mAP = float('nan')
    return head, acc, mAP, (va, pred.cpu().numpy(), yv_np)

head_d, acc_d, map_d, diag_d = train_linear(Xd, yd, spd)
head_y, acc_y, map_y, diag_y = train_linear(Xy, yy, spy)
print(f'val accuracy       — DINOv2: {acc_d:.3f} | YOLO: {acc_y:.3f}')
print(f'val mAP (macro AP) — DINOv2: {map_d:.3f} | YOLO: {map_y:.3f}')
print('note: this mAP is the RECOGNIZER (classification) mean Average Precision. Detection')
print('      mAP@0.50:0.95 is an IoU metric and is reported by the YOLO detector steps (2.4 / 4.2).')

winner = 'dino' if acc_d >= acc_y else 'yolo'
head, X, y, diag = (head_d, Xd, yd, diag_d) if winner == 'dino' else (head_y, Xy, yy, diag_y)
print('winning backbone:', winner)

torch.save({'state_dict': head.state_dict(), 'backbone': winner,
            'in_dim': X.shape[1], 'classes': FINE_NAMES}, b.root / 'recognizer.pt')
b._note('recognizer.pt'); b.save()
print('saved recognizer.pt (backbone =', winner, ')')

### 3.3 — Diagnose: per-class accuracy + confusion matrix

The point of Step 3. We rank classes by accuracy to surface weak SKUs, and plot the confusion matrix to see
which classes bleed into which (expect visually similar apples / milk cartons to confuse). We separate *low
support* (too few examples) from *genuinely confused* — Step 4 should target the right ones.

In [ ]:
import matplotlib.pyplot as plt
va, pred, ytrue = diag
present = sorted(set(ytrue) | set(pred))
name = lambda i: FINE_NAMES[i]

# Per-class accuracy + support
per_class = []
for c in present:
    m = ytrue == c; n = int(m.sum())
    acc = float((pred[m] == c).mean()) if n else float('nan')
    per_class.append({'fine': name(c), 'support': n, 'accuracy': round(acc, 3)})
per_class_df = pd.DataFrame(per_class)

worst10 = per_class_df.sort_values(['accuracy', 'support']).head(10)
best10 = per_class_df.sort_values(['accuracy', 'support'], ascending=[False, False]).head(10)
print('WEAKEST 10 classes:'); print(worst10.to_string(index=False))
print('\nBEST 10 classes:');   print(best10.to_string(index=False))

# Full confusion matrix over present classes
idx = {c: k for k, c in enumerate(present)}
cm = np.zeros((len(present), len(present)), int)
for t, p in zip(ytrue, pred):
    cm[idx[t], idx[p]] += 1

# Two confusion matrices: the K best classes (left) and the K worst classes (right)
name2c = {name(c): c for c in present}
K = min(14, len(present))
best_cls = [name2c[f] for f in
            per_class_df.sort_values(['accuracy', 'support'], ascending=[False, False]).head(K).fine]
worst_cls = [name2c[f] for f in per_class_df.sort_values(['accuracy', 'support']).head(K).fine]

def plot_cm(ax, cls_list, title):
    sub = [idx[c] for c in cls_list]
    m = cm[np.ix_(sub, sub)]                 # class-vs-class submatrix among the selected classes
    labels = [name(c) for c in cls_list]
    ax.set_facecolor('white')
    im = ax.imshow(m, cmap='Blues')
    ax.set_xticks(range(len(sub))); ax.set_yticks(range(len(sub)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(title)
    thr = m.max() / 2 if m.max() else 1
    for a in range(len(sub)):
        for bcol in range(len(sub)):
            if m[a, bcol]:
                ax.text(bcol, a, m[a, bcol], ha='center', va='center', fontsize=6,
                        color='white' if m[a, bcol] > thr else '#222')
    return im

fig, (axL, axR) = plt.subplots(1, 2, figsize=(2 * (0.5 * K + 3), 0.5 * K + 3), facecolor='white')
plot_cm(axL, best_cls, f'{K} BEST classes')
plot_cm(axR, worst_cls, f'{K} WORST classes')
fig.suptitle('Confusion matrices (val crops)')
plt.tight_layout(); plt.show()

b.put_table('diagnostics_per_class.csv', per_class_df)

### 3.4 — Explanation heatmap

A saliency/CAM-style heatmap over a few crops, to confirm the recognizer looks at the *product* and not the
background or a hand. We use a simple occlusion-sensitivity map (backbone-agnostic: slide a gray patch and
watch the winning-class score drop) so it works whichever backbone won.

In [ ]:
embed_fn = dino_embed if winner=='dino' else yolo_embed

@torch.no_grad()
def class_score(pil, cls):
    f = torch.tensor(embed_fn(pil)).unsqueeze(0).to(DEVICE)
    return torch.softmax(head(f), 1)[0, cls].item()

def occlusion_map(path, cls, grid=8, patch_frac=0.25):
    im = Image.open(path).convert('RGB'); W,H = im.size
    base = class_score(im, cls)
    pw, ph = int(W*patch_frac), int(H*patch_frac)
    heat = np.zeros((grid, grid))
    for gy in range(grid):
        for gx in range(grid):
            occ = im.copy()
            x = int(gx/grid*W); yv_ = int(gy/grid*H)
            for xx in range(x, min(x+pw,W)):
                for yy in range(yv_, min(yv_+ph,H)):
                    occ.putpixel((xx,yy),(128,128,128))
            heat[gy,gx] = base - class_score(occ, cls)     # drop in score = importance
    return im, heat

samples = crops_manifest.sample(min(3, len(crops_manifest)), random_state=0)
fig, axes = plt.subplots(1, len(samples), figsize=(5*len(samples), 5))
for ax, r in zip(np.atleast_1d(axes), samples.itertuples(index=False)):
    im, heat = occlusion_map(r.crop_path, fine2id[r.fine], grid=6)
    ax.imshow(im); ax.imshow(np.kron(heat, np.ones((im.size[1]//6, im.size[0]//6))),
                             cmap='jet', alpha=0.45, extent=(0,im.size[0],im.size[1],0))
    ax.set_title(f'{r.fine}'); ax.axis('off')
plt.tight_layout(); plt.show()

### 3.5 — Weak-class report (input to Step 4)

We name the classes Step 4 should augment: low accuracy AND/OR low support. This list is written to the
bundle so Step 4 reads it directly.

In [ ]:
ACC_FLOOR, SUPPORT_FLOOR = 0.6, 8
weak = per_class_df[(per_class_df.accuracy < ACC_FLOOR) | (per_class_df.support < SUPPORT_FLOOR)]
weak_report = weak.assign(
    reason=np.where(weak.support < SUPPORT_FLOOR, 'low_support', 'confused')
).reset_index(drop=True)
b.put_table('weak_class_report.csv', weak_report); b.save()
print(f'{len(weak_report)} weak classes flagged for Step 4:')
print(weak_report.to_string(index=False))
print('Step 3 complete.')

---
# Step 4 — Weak-class augmentation + detector re-tune

**Goal:** use Step 3's weak-class report to generate more samples for the weak SKUs, re-tune YOLO11, and
confirm a **real** validation lift.

Per the plan (decision 3), we use *any* generative method that lifts validation — not strictly a cGAN. We
start with cheap, reliable **copy-paste augmentation** (which for detection often rivals heavier generative
methods), and outline the optional conditional-generative upgrade. Success is judged only on a fixed
held-out set of **real** crops — never on generated data.

### 4.0 — Load the Step 3 checkpoint and weak-class report

In [ ]:
import os, pathlib, shutil
import numpy as np, pandas as pd, torch
from PIL import Image, ImageEnhance

assert 'b' in dir(), 'Run Step 1 setup cell first.'
weak_report = b.get_table('weak_class_report.csv')
boxes_manifest = b.get_table('boxes_manifest.csv')
kept_df = boxes_manifest[boxes_manifest.kept].reset_index(drop=True)
weak_classes = list(weak_report.fine)
print('weak classes to augment:', weak_classes if weak_classes else '(none — model is balanced)')

SCENES_DIR = pathlib.Path('/content/scenes')     # reuse Step 2's scene folder
CANVAS = 640
from ultralytics import YOLO

### 4.1 — Generate weak-class samples (cGAN)

Copy-paste augmentation of the real weak-class crops gave **no validation lift**, so we replace it with a **class-conditioned GAN**. A compact cGAN is trained on the weak-class subject crops, then samples new synthetic images for each weak SKU. Those generated crops are composed into fresh multi-item scenes (guaranteed to contain the weak class, plus real distractors), each pasted crop getting a proportional class-`0` (product) box — consistent with the single-class detector. Knobs: `CVCART_GAN_STEPS`, `CVCART_GEN_PER_CLASS`, `CVCART_AUG_SCENES`.

In [ ]:
import os, torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

AUG_DIR = b.root / 'aug'; AUG_DIR.mkdir(exist_ok=True)
(SCENES_DIR / 'images' / 'train').mkdir(parents=True, exist_ok=True)
(SCENES_DIR / 'labels' / 'train').mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
rng = np.random.default_rng(1)

GAN_STEPS       = int(os.environ.get('CVCART_GAN_STEPS', '600'))       # cGAN training iterations
GEN_PER_CLASS   = int(os.environ.get('CVCART_GEN_PER_CLASS', '24'))    # synthetic crops per weak class
PER_WEAK_SCENES = int(os.environ.get('CVCART_AUG_SCENES', '10'))       # scenes to compose per weak class
GAN_RES, Z_DIM  = 64, 64

by_class = {f: g for f, g in kept_df.groupby('fine')}
weak = [w for w in weak_classes if w in by_class]

def subject_crop(row):
    im = Image.open(row.abs_path).convert('RGB'); W, H = im.size
    x1, y1 = int(row.x * W), int(row.y * H); x2, y2 = int((row.x + row.w) * W), int((row.y + row.h) * H)
    c = im.crop((max(0, x1), max(0, y1), min(W, x2), min(H, y2)))
    return c if (c.size[0] >= 2 and c.size[1] >= 2) else im

# ---- conditional GAN trained on the weak-class subject crops (replaces copy-paste, which gave no lift) ----
class WeakCrops(Dataset):
    def __init__(self):
        self.rows = kept_df[kept_df.fine.isin(weak)].reset_index(drop=True)
        self.tf = T.Compose([T.ToTensor(), T.Normalize([0.5] * 3, [0.5] * 3)])
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows.iloc[i]
        return self.tf(subject_crop(r).resize((GAN_RES, GAN_RES))), w2i[r.fine]

class Generator(nn.Module):
    def __init__(self, z_dim, n_classes):
        super().__init__()
        self.label = nn.Embedding(n_classes, 16)
        self.fc = nn.Linear(z_dim + 16, 256 * 8 * 8)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh())
    def forward(self, z, y):
        return self.net(self.fc(torch.cat([z, self.label(y)], 1)).view(len(z), 256, 8, 8))

class Discriminator(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.label = nn.Embedding(n_classes, GAN_RES * GAN_RES)
        self.net = nn.Sequential(
            nn.Conv2d(4, 64, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Flatten(), nn.Linear(256 * 8 * 8, 1))
    def forward(self, x, y):
        lm = self.label(y).view(len(y), 1, GAN_RES, GAN_RES)
        return self.net(torch.cat([x, lm], 1)).squeeze(1)

w2i = {w: i for i, w in enumerate(weak)}
ds = WeakCrops()
made, gen = 0, {}
if not weak or len(ds) == 0:
    print('no weak classes / crops available — skipping cGAN augmentation')
else:
    bs = min(16, len(ds))
    loader = DataLoader(ds, batch_size=bs, shuffle=True, drop_last=len(ds) >= 2 * bs)
    G, D = Generator(Z_DIM, len(weak)).to(device), Discriminator(len(weak)).to(device)
    og = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5, 0.999))
    od = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5, 0.999))
    it = iter(loader)
    for step in range(GAN_STEPS):
        try:
            real, y = next(it)
        except StopIteration:
            it = iter(loader); real, y = next(it)
        real, y = real.to(device), y.to(device)
        z = torch.randn(len(real), Z_DIM, device=device); fake = G(z, y).detach()
        dl = (F.binary_cross_entropy_with_logits(D(real, y), torch.ones(len(real), device=device)) +
              F.binary_cross_entropy_with_logits(D(fake, y), torch.zeros(len(real), device=device)))
        od.zero_grad(); dl.backward(); od.step()
        z = torch.randn(len(real), Z_DIM, device=device); fake = G(z, y)
        gl = F.binary_cross_entropy_with_logits(D(fake, y), torch.ones(len(real), device=device))
        og.zero_grad(); gl.backward(); og.step()
        if (step + 1) % 150 == 0:
            print(f'cGAN step {step + 1}/{GAN_STEPS}: D {float(dl):.3f}  G {float(gl):.3f}')

    # sample synthetic crops per weak class
    G.eval()
    with torch.no_grad():
        for w in weak:
            z = torch.randn(GEN_PER_CLASS, Z_DIM, device=device)
            yy = torch.full((GEN_PER_CLASS,), w2i[w], device=device, dtype=torch.long)
            for t in G(z, yy).cpu():
                arr = (((t * 0.5 + 0.5).clamp(0, 1)) * 255).permute(1, 2, 0).numpy().astype('uint8')
                gen.setdefault(w, []).append(Image.fromarray(arr))

    # preview a few cGAN-generated samples
    import matplotlib.pyplot as plt
    show_w = weak[:min(4, len(weak))]; ncol = 6
    fig, axes = plt.subplots(len(show_w), ncol, figsize=(ncol * 1.5, len(show_w) * 1.6), facecolor='white')
    axes = np.atleast_2d(axes)
    for ri, wname in enumerate(show_w):
        for ci in range(ncol):
            axes[ri, ci].imshow(gen[wname][ci % len(gen[wname])]); axes[ri, ci].axis('off')
        axes[ri, 0].set_title(wname.replace('-', ' '), fontsize=8, loc='left', color='#0a7d3c')
    fig.suptitle('cGAN-augmented samples (weak classes)'); plt.tight_layout(); plt.show()

    # compose scenes: one GAN-generated weak instance + a few real distractors (exact class-0 boxes)
    others = list(by_class)
    for w in weak:
        for k in range(PER_WEAK_SCENES):
            canvas = np.full((CANVAS, CANVAS, 3), 245, np.uint8); gt = []
            gcrop = gen[w][int(rng.integers(0, len(gen[w])))]
            size = int(rng.integers(90, 200)); gc = gcrop.resize((size, size))
            px = int(rng.integers(0, CANVAS - size)); py = int(rng.integers(0, CANVAS - size))
            canvas[py:py + size, px:px + size] = np.array(gc)
            gt.append(((px + size / 2) / CANVAS, (py + size / 2) / CANVAS, size / CANVAS, size / CANVAS))
            for fine in rng.choice(others, size=int(rng.integers(1, 4)), replace=True):
                src = by_class[fine].iloc[int(rng.integers(0, len(by_class[fine])))]
                s2 = int(rng.integers(90, 200)); im = subject_crop(src).resize((s2, s2))
                qx = int(rng.integers(0, CANVAS - s2)); qy = int(rng.integers(0, CANVAS - s2))
                canvas[qy:qy + s2, qx:qx + s2] = np.array(im)
                gt.append(((qx + s2 / 2) / CANVAS, (qy + s2 / 2) / CANVAS, s2 / CANVAS, s2 / CANVAS))
            stem = f'gan_{w}_{k:02d}'
            Image.fromarray(canvas).save(SCENES_DIR / 'images' / 'train' / f'{stem}.jpg')
            (SCENES_DIR / 'labels' / 'train' / f'{stem}.txt').write_text(
                '\n'.join(f'0 {xc:.6f} {yc:.6f} {wd:.6f} {hd:.6f}' for (xc, yc, wd, hd) in gt) + '\n')
            made += 1

print(f'cGAN: {sum(len(v) for v in gen.values())} synthetic crops; composed {made} weak-class scenes '
      f'across {len(weak)} weak classes')

### 4.2 — Re-tune the detector and measure the lift

We re-tune YOLO11 on the augmented scene set and compare mAP on the **same held-out val split** as Step 2.
We keep `detector_v2.pt` only if it doesn't regress; the augmentation earns its place by the numbers.

In [ ]:
EPOCHS = int(os.environ.get('CVCART_EPOCHS2', '40'))
det2 = YOLO('yolo11s.pt')
res2 = det2.train(data=str(SCENES_DIR/'data.yaml'), epochs=EPOCHS, imgsz=768,
                  patience=15, seed=0, device=0 if torch.cuda.is_available() else 'cpu',
                  mosaic=1.0, mixup=0.15, degrees=10, scale=0.5, perspective=0.0005, flipud=0.5, verbose=False)

# Compare against the Step 2 detector on the current val split.
base = YOLO(str(b.root/'detector.pt'))
m_old = base.val(data=str(SCENES_DIR/'data.yaml'), verbose=False)
m_new = det2.val(data=str(SCENES_DIR/'data.yaml'), verbose=False)
print(f'mAP50  before: {float(m_old.box.map50):.3f} -> after: {float(m_new.box.map50):.3f}')
print(f'mAP50-95 before: {float(m_old.box.map):.3f} -> after: {float(m_new.box.map):.3f}')

new_path = os.path.join(getattr(res2,'save_dir','runs/detect/train2'),'weights','best.pt')
if float(m_new.box.map50) >= float(m_old.box.map50):
    shutil.copy(new_path, b.root/'detector_v2.pt'); b._note('detector_v2.pt'); b.save()
    print('kept detector_v2.pt (lift confirmed)')
else:
    shutil.copy(b.root/'detector.pt', b.root/'detector_v2.pt'); b._note('detector_v2.pt'); b.save()
    print('no lift — detector_v2.pt falls back to the Step 2 detector')
print('Step 4 complete.')

---
# Step 5 — Gradio self-checkout demo

**Goal:** a working demo. Upload a composed multi-class, multi-item photo; the app **detects** products,
**recognizes** each, **matches** the catalog, and **totals** the basket from the price table — loading only
saved checkpoints, no training at demo time.

### 5.0 — Load every checkpoint

Detector (`detector_v2.pt`), recognizer (`recognizer.pt` + its winning backbone), catalog gallery, and the
price table.

In [ ]:
import pathlib, numpy as np, pandas as pd, torch
from PIL import Image
from ultralytics import YOLO

assert 'b' in dir(), 'Run Step 1 setup cell first.'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

detector = YOLO(str(b.root/'detector_v2.pt'))
ckpt = torch.load(b.root/'recognizer.pt', map_location=DEVICE)
FINE_NAMES = ckpt['classes']; BACKBONE = ckpt['backbone']
head = torch.nn.Linear(ckpt['in_dim'], len(FINE_NAMES)); head.load_state_dict(ckpt['state_dict'])
head = head.to(DEVICE).eval()

catalog_embeddings = b.get_array('catalog_embeddings.npy')
catalog_index = b.get_table('catalog_index.csv')
price_table = b.get_table('price_table.csv')
price_of = dict(zip(price_table.fine, price_table.price_usd))
print('loaded checkpoints | recognizer backbone:', BACKBONE)

# Rebuild the winning feature extractor (must match Step 3 exactly).
import torchvision.transforms as T
if BACKBONE == 'dino':
    if 'dino' not in dir():
        dino = torch.hub.load('facebookresearch/dinov2','dinov2_vits14').to(DEVICE).eval()
        for p in dino.parameters(): p.requires_grad_(False)
    dtf = T.Compose([T.Resize(256),T.CenterCrop(224),T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    @torch.no_grad()
    def embed(pil):
        x = dtf(pil.convert('RGB')).unsqueeze(0).to(DEVICE)
        return torch.nn.functional.normalize(dino(x),dim=1).squeeze(0).cpu().numpy()
else:
    ym = detector.model.to(DEVICE).eval()
    @torch.no_grad()
    def embed(pil):
        import torch.nn.functional as Fn
        x = T.Compose([T.Resize((224,224)),T.ToTensor()])(pil.convert('RGB')).unsqueeze(0).to(DEVICE)
        feats=[]; h=ym.model[-2].register_forward_hook(lambda m,i,o: feats.append(o))
        try: ym(x)
        finally: h.remove()
        f = feats[-1]; f = f[-1] if isinstance(f,(list,tuple)) else f
        return Fn.normalize(Fn.adaptive_avg_pool2d(f,1).flatten(1),dim=1).squeeze(0).cpu().numpy()

### 5.1 — The checkout pipeline

Per uploaded image: detect boxes → crop → recognize each crop → nearest-neighbor catalog match (for a "this
is what we think it is" panel) → look up price → itemized basket + total. Low-confidence items are flagged,
echoing Step 3's "silent misreads are worse than useless" principle.

In [ ]:
from PIL import ImageDraw
import os
DET_CONF  = float(os.environ.get('CVCART_DET_CONF', '0.10'))   # low: single-class detector is weak (mAP~0.3)
DET_IMGSZ = int(os.environ.get('CVCART_DET_IMGSZ', '768'))     # larger input -> better recall
DET_IOU   = float(os.environ.get('CVCART_DET_IOU', '0.55'))     # NMS IoU: lower = suppress overlapping dup boxes harder

@torch.no_grad()
def recognize(crop):
    f = torch.tensor(embed(crop)).unsqueeze(0).to(DEVICE)
    probs = torch.softmax(head(f), 1)[0]
    cls = int(probs.argmax()); conf = float(probs[cls])
    return FINE_NAMES[cls], conf

def detect_and_recognize(image):
    """Detect boxes -> recognize each crop. Returns (rgb_image, [det dicts]); NO confidence filter
    here (the kiosk slider applies the threshold at display time, so it re-prices without re-running)."""
    im = image.convert('RGB')
    res = detector(np.array(im), imgsz=DET_IMGSZ, conf=DET_CONF, iou=DET_IOU,
                   agnostic_nms=True, augment=True, verbose=False)[0]
    boxes = res.boxes.xyxy.cpu().numpy()
    if len(boxes) == 0:                                   # weak detector / single product -> whole image
        boxes = np.array([[0, 0, im.width, im.height]], dtype='float32')
    dets = []
    for box in boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        if x2 - x1 < 8 or y2 - y1 < 8:
            continue
        fine, conf = recognize(im.crop((x1, y1, x2, y2)))
        dets.append({'box': (x1, y1, x2, y2), 'item': fine, 'confidence': conf,
                     'unit_price': float(price_of.get(fine, 0.0))})
    return im, dets

# quick smoke test on a Step 2 scene if present
import glob
demo_imgs = glob.glob('/content/scenes/images/val/*.jpg')
if demo_imgs:
    _im, _dets = detect_and_recognize(Image.open(demo_imgs[0]))
    print('smoke test — detections:', len(_dets))

### 5.2 — Launch the Gradio app

Upload a composed basket photo and read off the itemized total. `share=True` gives a public link from Colab.

In [ ]:
import gradio as gr
from datetime import datetime

STORE_NAME = 'FreshMart'
DEFAULT_THR_PCT = 30      # 81-class softmax rarely exceeds ~0.5, so a 70% default billed nothing

def render_annotated(base_im, dets, thr):
    im = base_im.copy(); d = ImageDraw.Draw(im)
    for det in dets:
        x1, y1, x2, y2 = det['box']; kept = det['confidence'] >= thr
        col = '#20c060' if kept else '#9aa0a6'
        d.rectangle([x1, y1, x2, y2], outline=col, width=3)
        d.text((x1, max(0, y1 - 12)), f"{det['item']} {det['confidence']:.2f}", fill=col)
    return im

def _kept_total(dets, thr):
    kept = [x for x in dets if x['confidence'] >= thr]
    return kept, sum(x['unit_price'] for x in kept)

def render_ticker(dets, thr):         # system-info bar shown flush above the receipt
    kept, _ = _kept_total(dets, thr)
    return ("<div class='ticker'><span class='t-dot'></span>"
            + str(len(kept)) + " of " + str(len(dets)) + " detected item(s) billed"
            + " &nbsp;·&nbsp; confidence threshold " + str(int(round(thr * 100))) + "%</div>")

def render_receipt(dets, thr):
    kept, total = _kept_total(dets, thr)
    agg = {}
    for x in kept:
        a = agg.setdefault(x['item'], {'qty': 0, 'unit': x['unit_price']})
        a['qty'] += 1
    rows = ''
    for name in sorted(agg):
        v = agg[name]; sub = v['qty'] * v['unit']
        desc = name.replace('-', ' ')
        rows += ("<tr><td class='r-name'>" + desc + "</td><td class='r-c'>" + str(v['qty']) +
                 "</td><td class='r-r'>$" + f"{v['unit']:.2f}" + "</td><td class='r-r'>$" +
                 f"{sub:.2f}" + "</td></tr>")
    if not rows:
        rows = ("<tr><td colspan='4' class='r-empty'>🛒 Please scan an item from your basket "
                "to add it to your receipt</td></tr>")
    now = datetime.now().strftime('%a %d %b %Y  %H:%M')
    return (
        "<div class='receipt'>"
        "<div class='r-store'>🛒 " + STORE_NAME + "</div>"
        "<div class='r-sub'>SELF-CHECKOUT KIOSK</div>"
        "<div class='r-meta'>" + now + "</div>"
        "<div class='r-dash'></div>"
        "<table class='r-tbl'><thead><tr><th>Item</th><th class='r-c'>Qty</th>"
        "<th class='r-r'>Unit</th><th class='r-r'>Amount</th></tr></thead><tbody>"
        + rows +
        "</tbody></table>"
        "<div class='r-dash'></div>"
        "<div class='r-total'><span>TOTAL</span><span>$" + f"{total:.2f}" + "</span></div>"
        "<div class='r-thanks'>✱ ✱ ✱ &nbsp; THANK YOU &nbsp; ✱ ✱ ✱</div>"
        "</div>"
    )

def render_panel(dets, thr):          # ticker + receipt in ONE block -> no gap between them
    return render_ticker(dets, thr) + render_receipt(dets, thr)

def payment_html(total, n_items):
    now = datetime.now().strftime('%a %d %b %Y  %H:%M:%S')
    return (
        "<div class='pay-ok'>"
        "<div class='pay-check'>✓</div>"
        "<div class='pay-title'>PAYMENT SUCCESSFUL</div>"
        "<div class='pay-amt'>$" + f"{total:.2f}" + "</div>"
        "<div class='pay-line'>" + str(n_items) + " item(s) &nbsp;·&nbsp; " + now + "</div>"
        "<div class='pay-line'>Please collect your items — thank you for shopping at " + STORE_NAME + "!</div>"
        "</div>"
    )

def scan(image, thr_pct):             # fires automatically when a photo is uploaded/captured
    thr = thr_pct / 100.0
    if image is None:
        return None, None, render_panel([], thr)
    base_im, dets = detect_and_recognize(image)
    return render_annotated(base_im, dets, thr), (base_im, dets), render_panel(dets, thr)

def reprice(state, thr_pct):          # slider moved -> re-filter + re-price WITHOUT re-detecting
    thr = thr_pct / 100.0
    if not state:
        return None, render_panel([], thr)
    base_im, dets = state
    return render_annotated(base_im, dets, thr), render_panel(dets, thr)

def do_pay(state, thr_pct):           # Pay -> confirmation panel + reveal "New Customer"
    thr = thr_pct / 100.0
    if not state:
        return gr.update(value="<div class='pay-warn'>Nothing to pay — snap a basket first.</div>", visible=True), gr.update(visible=False)
    _base, dets = state
    kept, total = _kept_total(dets, thr)
    if not kept:
        return gr.update(value="<div class='pay-warn'>No items above the threshold to charge.</div>", visible=True), gr.update(visible=False)
    return gr.update(value=payment_html(total, len(kept)), visible=True), gr.update(visible=True)

def new_customer():                   # reset the kiosk for the next shopper
    thr = DEFAULT_THR_PCT / 100.0
    return (None, None, None, render_panel([], thr),
            gr.update(value='', visible=False), gr.update(visible=False))

CSS = """
/* --- force BRIGHT (day) theme regardless of the viewer's system/dark setting --- */
:root, .dark{
  --body-background-fill:#f4f6f8 !important; --body-text-color:#1a1a1a !important;
  --body-text-color-subdued:#5a6270 !important; --background-fill-primary:#ffffff !important;
  --background-fill-secondary:#eef1f4 !important; --block-background-fill:#ffffff !important;
  --block-label-text-color:#0a7d3c !important; --block-title-text-color:#0a7d3c !important;
  --border-color-primary:#e2e5ea !important; --input-background-fill:#ffffff !important;
  --input-border-color:#d5dae1 !important; --button-secondary-background-fill:#eef1f4 !important;
  --button-secondary-text-color:#222 !important; --table-even-background-fill:#ffffff !important;
  --table-odd-background-fill:#f7f9fb !important; color-scheme:light !important;
}
.gradio-container, gradio-app{background:#f4f6f8 !important}
/* slider label -> visible brand green */
.k-field label, .k-field span, .k-field .block-info{color:#0a7d3c !important;font-weight:600 !important}
/* image section headers: WHITE text on a green bar (readable, not dim) */
.field-hd{background:#0a7d3c;color:#fff !important;font-weight:700;font-size:.92em;letter-spacing:.3px;
  padding:7px 12px;border-radius:8px 8px 0 0;text-align:center;margin-bottom:0}
.field-hd *{color:#fff !important}
/* MOBILE ONLY: enlarge the snap/upload section ~2x */
@media (max-width:820px){
  .k-upload{min-height:232px !important}
  .k-upload .image-container, .k-upload [data-testid='image'], .k-upload .upload-container,
  .k-upload .wrap, .k-upload img, .k-upload video, .k-upload canvas{min-height:232px !important;max-height:320px !important}
}
#kiosk-row{gap:16px}
.kiosk-hero{text-align:center;padding:16px;border-radius:16px;color:#fff;margin-bottom:6px;
  background:linear-gradient(135deg,#0a7d3c,#12a454)}
.kiosk-hero, .kiosk-hero *{color:#fff !important}
.kiosk-logo{font-size:2.1em;font-weight:800;letter-spacing:1px}
.kiosk-tag{opacity:.92;font-size:.95em}
.ticker{font-family:'Courier New',monospace;font-size:.8em;color:#2a3340 !important;background:#e9f4ec !important;
  border:1px solid #cfe6d5;border-bottom:none;border-radius:8px 8px 0 0;padding:7px 12px;margin:0 auto;
  text-align:center;font-weight:700}
.ticker .t-dot{display:inline-block;width:8px;height:8px;border-radius:50%;background:#20c060;
  margin-right:7px;vertical-align:middle}
.receipt{font-family:'Courier New',monospace;background:#fff !important;color:#141414 !important;font-size:11px;
  max-width:100%;margin:0 auto;padding:14px 18px;border-radius:0 0 8px 8px;box-shadow:0 3px 16px rgba(0,0,0,.14)}
.receipt, .receipt table, .receipt th, .receipt td, .receipt div, .receipt span{color:#141414 !important}
.receipt .r-sub, .receipt .r-meta{color:#666 !important}
.receipt .r-store{text-align:center;font-size:1.5em;font-weight:800;letter-spacing:2px}
.receipt .r-sub{text-align:center;font-size:.8em;letter-spacing:2px}
.receipt .r-meta{text-align:center;font-size:.8em;margin-top:3px}
.receipt .r-dash{border-top:2px dashed #bcbcbc;margin:11px 0}
.receipt .r-tbl{width:100%;border-collapse:collapse;font-size:.9em}
.receipt .r-tbl th{text-align:left;padding:2px 0;border-bottom:1px solid #ddd;font-weight:700}
.receipt .r-tbl td{padding:3px 0;vertical-align:top}
.receipt .r-c{text-align:center;white-space:nowrap}
.receipt .r-r{text-align:right;white-space:nowrap}
.receipt .r-name{text-transform:capitalize}
.receipt .r-empty{text-align:center;padding:12px 4px;color:#0a7d3c !important;font-weight:700}
.receipt .r-total{display:flex;justify-content:space-between;font-weight:800;font-size:1.25em;margin-top:4px}
.receipt .r-thanks{text-align:center;margin-top:9px;font-size:.85em;letter-spacing:1px}
.pay-ok{max-width:100%;margin:12px auto 0;text-align:center;padding:18px;border-radius:12px;color:#fff;
  background:linear-gradient(135deg,#0a7d3c,#12a454);box-shadow:0 3px 16px rgba(0,0,0,.18)}
.pay-ok *{color:#fff !important}
.pay-check{font-size:2.4em;line-height:1}
.pay-title{font-weight:800;letter-spacing:2px;margin-top:4px}
.pay-amt{font-size:2em;font-weight:800;margin:6px 0}
.pay-line{font-size:.82em;opacity:.95;margin-top:3px}
.pay-warn{max-width:100%;margin:12px auto 0;text-align:center;padding:12px;border-radius:10px;
  background:#fff3cd !important;color:#7a5b00 !important;border:1px solid #ffe08a}
@media (max-width:820px){#kiosk-row{flex-direction:column !important}.receipt,.pay-ok,.pay-warn,.ticker{max-width:100%}}
"""

with gr.Blocks(title=STORE_NAME + ' Self-Checkout', css=CSS,
               theme=gr.themes.Soft(primary_hue='green')) as demo:
    gr.HTML("<div class='kiosk-hero'><div class='kiosk-logo'>🛒 " + STORE_NAME + "</div>"
            "<div class='kiosk-tag'>Self-Checkout — snap your basket, it rings up automatically</div></div>")
    state = gr.State()
    with gr.Row(elem_id='kiosk-row'):
        with gr.Column(scale=6):          # LEFT: small input stacked above detected (more room)
            gr.HTML("<div class='field-hd'>📷 Snap or upload your basket</div>")
            inp = gr.Image(type='pil', sources=['upload', 'webcam'], elem_classes='k-upload',
                           show_label=False, height=113)
            gr.HTML("<div class='field-hd'>Detected items</div>")
            out_img = gr.Image(type='pil', show_label=False, height=230, interactive=False)
        with gr.Column(scale=4):          # RIGHT: ticker+receipt (one block) -> pay
            out_panel = gr.HTML(render_panel([], DEFAULT_THR_PCT / 100.0))
            pay_btn = gr.Button('💳  Pay', variant='primary', size='lg')
            pay_panel = gr.HTML(visible=False)
            new_btn = gr.Button('🔄  New Customer', visible=False)
    with gr.Accordion('⚙  Settings', open=False):   # collapsed by default
        gr.HTML("<div class='field-hd'>Confidence threshold (%) — detected items below this are NOT billed</div>")
        conf = gr.Slider(0, 100, value=DEFAULT_THR_PCT, step=5, elem_classes='k-field', show_label=False)

    inp.change(scan, [inp, conf], [out_img, state, out_panel])   # auto-detect on upload
    conf.change(reprice, [state, conf], [out_img, out_panel])
    pay_btn.click(do_pay, [state, conf], [pay_panel, new_btn])
    new_btn.click(new_customer, None, [inp, out_img, state, out_panel, pay_panel, new_btn])
    if demo_imgs:
        gr.Examples([[demo_imgs[0]]], inputs=inp)

demo.launch(share=True, prevent_thread_lock=True, debug=False)

# --- QR code so a phone can scan and open the kiosk ---
url = getattr(demo, 'share_url', None) or getattr(demo, 'local_url', None)
print('📱 Open on your phone:', url)
try:
    import qrcode
    from IPython.display import display
    display(qrcode.make(url).resize((240, 240)))
except Exception as e:
    print('(QR skipped:', e, '- scan the URL above manually)')

---
## Summary

This notebook builds an end-to-end computer-vision self-checkout in five steps. **Step 1** turns the label-only
GroceryStoreDataset into reusable assets — a labels table, a frozen-DINOv2 catalog gallery, and a price table.
**Step 2** manufactures bounding boxes with an auto-boxer (the dataset ships none), composes multi-item scenes,
and fine-tunes YOLO11 as a single-class `product` detector that localizes *where* items are. **Step 3** trains a
tiny linear recognizer on frozen features, benchmarks the DINOv2 vs YOLO backbone, and diagnoses per-class
accuracy and confusions. **Step 4** targets the weakest classes with cGAN-generated samples and re-tunes the
detector, keeping the new weights only if validation improves. **Step 5** wires it together into a Gradio kiosk
that detects, recognizes, catalog-matches, prices, and totals a basket. Each step checkpoints to Google Drive so
it can be run and resumed independently.

**Honest notes:**

- Every bounding box is **manufactured by Step 2's auto-boxer**, so the reported mAP measures agreement with the
  auto-boxer, not true ground truth. Run the Step 2.6 gold-set evaluation for one trustworthy number before
  quoting mAP.
- The detector is trained on **composed synthetic scenes**; real uploads with heavy occlusion, overlap, or odd
  angles will score lower than the validation numbers suggest.
- The detector is **single-class** (localization only). Fine-grained 81-class recognition is the recognizer's
  job and remains the hardest part of the pipeline.
- The **cGAN samples are low-resolution (64px)** — they add appearance variety, but the lift is not guaranteed;
  Step 4.2's before/after mAP is the real test of whether they helped.
- Improvement priority: **better / hand-checked boxes → more diverse, realistic scenes → stronger recognizer
  backbone → targeted weak-class data.**